In [1]:
! pip install transformers


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:

import sys
sys.path.append('..')

from data.dataset import create_dataloader
from data.dataset01 import create_all_dataloaders

from torchvision import transforms
# Initialize model components
from helpers.image_encoder import ImageEncoder
from helpers.text_encoder import TextEncoder
from helpers.fusion import FusionModule
from helpers.projection import ProjectionModule
import torch
from model.model import ICDRetrievalModel 
from trainer.trainer import MultimodalContrastiveTrainer
import wandb
from transformers import AutoTokenizer, AutoModel



In [3]:
# Hyperparameters
BATCH_SIZE = 16
NUM_EPOCHS = 2
LEARNING_RATE = 1e-4
TEMPERATURE = 0.07
MIN_IMAGES_PER_STUDY = 2

# Example dimensions (adjust to your actual models)
IMG_DIM = 384  # DINO output dimension
TEXT_DIM = 768  # MedCLIP output dimension
CONCATENATED_DIM = IMG_DIM + TEXT_DIM # If using concatenation in fusion

EMBEDDING_DIM = 512  # Projected fused embedding -> can also try with 256 and see if it helps with contrastive learning



In [5]:
# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# # Transforms
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

# Create dataloaders with FLAT collate
print("Loading datasets...")
# train_loader = create_dataloader(
#     mimic_root="/datasets/MIMIC-CXR",  # UPDATE THIS
#     split='train',
#     batch_size=BATCH_SIZE,
#     min_images_per_study=MIN_IMAGES_PER_STUDY,
#     num_workers=4,
#     # transform=transform,
#     return_all_images=True,
#     use_flat_collate=True,  # IMPORTANT: Use flat collate for contrastive learning
#     shuffle=True,
# )

# val_loader = create_dataloader(
#     mimic_root="/datasets/MIMIC-CXR",  # UPDATE THIS
#     split='val',
#     batch_size=BATCH_SIZE,
#     min_images_per_study=MIN_IMAGES_PER_STUDY,
#     num_workers=4,
#     # transform=transform,
#     return_all_images=True,
#     use_flat_collate=True,  # IMPORTANT: Use flat collate
#     shuffle=False,
# )


# Create all loaders at once
train_loader, val_loader, test_loader = create_all_dataloaders(
    mimic_root="/datasets/MIMIC-CXR",
    split_ratios=(0.7, 0.15, 0.15),  # 70-15-15 split
    split_level='patient',  # or 'study'
    batch_size=16,
    use_flat_collate=True,
    seed=42,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")



Using device: cuda
Loading datasets...

Scanning /datasets/MIMIC-CXR/files for studies...


Found 96918 total studies with >= 2 images
Splitting data at patient level with ratios (0.7, 0.15, 0.15)...
Split: train
  Total studies: 68218
  Unique patients: 33993


Scanning /datasets/MIMIC-CXR/files for studies...
Found 96918 total studies with >= 2 images
Splitting data at patient level with ratios (0.7, 0.15, 0.15)...
Split: val
  Total studies: 14163
  Unique patients: 7284


Scanning /datasets/MIMIC-CXR/files for studies...
Found 96918 total studies with >= 2 images
Splitting data at patient level with ratios (0.7, 0.15, 0.15)...
Split: test
  Total studies: 14537
  Unique patients: 7285

Train batches: 4264
Val batches: 886
Test batches: 909


In [6]:
image_encoder_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')  # pretrained DINO backbone
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
text_encoder_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")   # pretrained MedCLIP text encoder

image_encoder = ImageEncoder(image_encoder_model)
text_encoder = TextEncoder(text_encoder_model, tokenizer)

Using cache found in /home/icdmultimodalproject/.cache/torch/hub/facebookresearch_dino_main


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [7]:
fusion_module = FusionModule(
    method='concat'  # or 'add', 'multiply'
)
projection_module = ProjectionModule(
    input_dim= CONCATENATED_DIM, # check if .output_dim is defined in fusion module, otherwise calculate based on fusion method
    output_dim=EMBEDDING_DIM
)

# We don't use ICD projection for retrieval, but keep it for compatibility
icd_projection_module = ProjectionModule(
    input_dim=TEXT_DIM,
    output_dim=EMBEDDING_DIM
)

model = ICDRetrievalModel(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    fusion_module=fusion_module,
    projection_module=projection_module,
    icd_projection_module=icd_projection_module,
    embedding_dim=EMBEDDING_DIM,
    fusion_method='concat'
)

# Optimizer (only train projection modules, encoders are frozen)
trainable_params = list(icd_projection_module.parameters()) + \
                    list(projection_module.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)

# # Scheduler
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, 
#     T_max=NUM_EPOCHS,
#     eta_min=1e-6
# )

# Initialize wandb (optional)
wandb.init(
    project="radiograph-retrieval",
    config={
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "temperature": TEMPERATURE,
        "num_epochs": NUM_EPOCHS,
    }
)

# Create trainer
trainer = MultimodalContrastiveTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    temperature=TEMPERATURE,
    log_wandb=True,  # Set to True if using wandb
)

# Train
print("\nStarting training...")
trainer.train(
    num_epochs=NUM_EPOCHS,
    save_dir='./checkpoints',
    save_every=5
)

print("\nTraining complete!")



wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/icdmultimodalproject/.netrc.


wandb: Currently logged in as: luca24ever (luca24ever_unibo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Starting training...

Epoch 1/2


Epoch 1:   0%|          | 4/4264 [01:33<27:33:50, 23.29s/it, loss=3.2306, avg_loss=2.9616]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f9920459850>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f9939b5cc50, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f9939b5c170, raw_cell="fusion_module = FusionModule(
    method='concat' .." transformed_cell="fusion_module = FusionModule(
    method='concat' .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bicd-server/home/icdmultimodalproject/project/src/task2/notebooks/main02.ipynb#W5sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost